In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import netCDF4
from dask import array as da
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import BoundaryNorm

import geopandas as gp
from geopy import distance

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from cartopy.util import add_cyclic_point

import glob
from tqdm import tqdm

import re
from scipy.stats import gmean
import cmocean

In [2]:
month_names = {
    1: 'January',
    2: 'February',
    3: 'March',
    4: 'April',
    5: 'May',
    6: 'June',
    7: 'July',
    8: 'August',
    9: 'September',
    10: 'October',
    11: 'November',
    12: 'December'
}

In [3]:
def calc_distance(x):
    return distance.distance(
        x[['OBS_CENTROID_LAT', 'OBS_CENTROID_LON']].values,
        x[['FCST_CENTROID_LAT', 'FCST_CENTROID_LON']].values).km

def analyze_pkl(df):
    cluster_pair_str = 'CF0*[1-9][0-9]*_CO0*[1-9][0-9]*'
    cluster_pairs = df[df['OBJECT_ID'].str.fullmatch(cluster_pair_str)].copy(deep=True)
    cluster_pairs['CLUSTER_IDX'] = cluster_pairs['OBJECT_ID'].str[-3:].astype(int)
    cluster_pairs['INTERSECTION_OVER_UNION'] = cluster_pairs['INTERSECTION_AREA'] / cluster_pairs['UNION_AREA']
    var_cols = [
        'CENTROID_DIST',
        'INTERSECTION_OVER_UNION',
    ]
    cols_shared = [
        'CLUSTER_IDX',
        'INIT',
        'MEMBER',
        'LEAD',
        'INIT_MONTH',
        'VALID_MONTH'
    ]
    df_pairs = cluster_pairs[np.concatenate([cols_shared, var_cols])]

    fcst_cluster_str = 'CF0*[1-9][0-9]*'
    obs_cluster_str = 'CO0*[1-9][0-9]*'
    fcst_clus_matched = df[df['OBJECT_ID'].str.fullmatch(fcst_cluster_str)].copy(deep=True)
    obs_clus_matched = df[df['OBJECT_ID'].str.fullmatch(obs_cluster_str)].copy(deep=True)
    fcst_clus_matched['CLUSTER_IDX'] = fcst_clus_matched['OBJECT_ID'].str.lstrip('CF0*').astype(int)
    obs_clus_matched['CLUSTER_IDX'] = obs_clus_matched['OBJECT_ID'].str.lstrip('CO0*').astype(int)
    cols = [
        'AREA',
        'INTENSITY_50',
        'CENTROID_LAT',
        'CENTROID_LON'
    ]
    fcst_cols = fcst_clus_matched[np.concatenate([cols_shared, cols])].rename(columns={col: 'FCST_'+col for col in cols})
    obs_cols = obs_clus_matched[np.concatenate([cols_shared, cols])].rename(columns={col: 'OBS_'+col for col in cols})
    df_clus = pd.merge(fcst_cols, obs_cols)
    for col in ['AREA', 'INTENSITY_50']:
        df_clus[f'LOG_{col}_RATIO'] = np.log(df_clus[f'FCST_{col}'] / df_clus[f'OBS_{col}'])
    df_clus['CENTROID_DIST_KM'] = df_clus.apply(lambda x: calc_distance(x), axis=1)
    df_clus['ABS_LOG_AREA_RATIO'] = abs(df_clus['LOG_AREA_RATIO'])
    df_clus['ABS_LOG_INTENSITY_50_RATIO'] = abs(df_clus['LOG_INTENSITY_50_RATIO'])

    return pd.merge(df_pairs.reset_index().drop(columns=['index']), df_clus)

In [4]:
MODE_output = '/glade/work/jtcohen/MODE_files_final/saved_output'
df_random = pd.read_pickle(f'{MODE_output}/mode_1989-2018_SMYLE_OISST_24L_ocetrac_random.pkl')
df = pd.read_pickle(f'{MODE_output}/mode_1989-2018_SMYLE_OISST_24L_ocetrac_r3.pkl')
# df = pd.read_pickle('/glade/work/jtcohen/MODE_files_final/output_r3/mode_1989-2018_SMYLE_FOSI_24L_ocetrac.pkl')

In [5]:
%%time
df_random_final = analyze_pkl(df_random)
df_final = analyze_pkl(df)

CPU times: user 1min 33s, sys: 189 ms, total: 1min 34s
Wall time: 1min 39s


In [7]:
df_final_10M = df_final[df_final['MEMBER']<=10].reset_index().drop(columns=['index'])

In [6]:
vars = ['CENTROID_DIST_KM', 'INTERSECTION_OVER_UNION']
log_vars = ['ABS_LOG_AREA_RATIO', 'ABS_LOG_INTENSITY_50_RATIO', 'LOG_AREA_RATIO', 'LOG_INTENSITY_50_RATIO']

In [7]:
ds_final_mean = df_final.groupby(['INIT_MONTH', 'LEAD'])[vars+log_vars].mean().to_xarray().rename(
    {'INIT_MONTH': 'month',
     'LEAD': 'lead'}
)

ds_random_mean = df_random_final.groupby(['INIT_MONTH', 'LEAD', 'MEMBER'])[vars+log_vars].mean().to_xarray().rename(
    {'INIT_MONTH': 'month',
     'LEAD': 'lead',
     'MEMBER': 'member'}
)

ds_final_mean_10M = df_final_10M.groupby(['INIT_MONTH', 'LEAD'])[vars+log_vars].mean().to_xarray().rename(
    {'INIT_MONTH': 'month',
     'LEAD': 'lead'}
)

In [ ]:
# qs = [.1, .25, .5, .75, .9]
# ds_final_qs = [df_final.groupby(['INIT_MONTH', 'LEAD'])[vars+log_vars].quantile(q).to_xarray().rename(
#     {'INIT_MONTH': 'month',
#      'LEAD': 'lead'}
# ) for q in qs]
# ds_random_qs = [df_random_final.groupby(['INIT_MONTH', 'LEAD'])[vars+log_vars].quantile(q).to_xarray().rename(
#     {'INIT_MONTH': 'month',
#      'LEAD': 'lead'}
# ) for q in qs]
# ds_qs = xr.concat(ds_final_qs, dim='q')
# ds_qs['q'] = qs
# ds_random_qs = xr.concat(ds_random_qs, dim='q')
# ds_random_qs['q'] = qs

In [9]:
GRL_code = '/glade/u/home/jtcohen/MODE/notebooks/GRL_code/final_data'
ds_final_mean.to_netcdf(f'{GRL_code}/leadtime_attrs_FOSI_r3.nc')
# ds_random_mean.to_netcdf(f'{GRL_code}/leadtime_attrs_random_bymember.nc')

In [35]:
ds_final_mean_10M.to_netcdf(f'{GRL_code}/leadtime_attrs_r3_10M.nc')

In [36]:
for rad in tqdm([2, 4, 5, 6, 7]):
    df_rad = pd.read_pickle(f'{MODE_output}/mode_1989-2018_SMYLE_OISST_24L_ocetrac_r{rad}.pkl')
    df_rad_final = analyze_pkl(df_rad)
    ds_rad_mean = df_rad_final.groupby(['INIT_MONTH', 'LEAD'])[vars+log_vars].mean().to_xarray().rename(
        {'INIT_MONTH': 'month',
         'LEAD': 'lead'}
    )
    ds_rad_mean.to_netcdf(f'{GRL_code}/leadtime_attrs_r{rad}_10M.nc')

100%|██████████| 5/5 [04:15<00:00, 51.12s/it]


# Attributes by size

In [35]:
df_final_og = df_final.copy()

In [36]:
len(set(list(df_final.groupby('LEAD')['FCST_AREA'].transform(lambda x: pd.qcut(x, 2, labels=labels)).values)))

45

In [49]:
n = 10
colors = plt.cm.RdBu_r(np.linspace(0,1,n))
labels = np.arange(10) 

# df_final['FCST_AREA_QUANTILE'] = pd.qcut(df_final['FCST_AREA'], n, labels=labels)
df_final['FCST_AREA_QUANTILE'] = df_final.groupby('LEAD')['FCST_AREA'].transform(lambda x: pd.qcut(x, n, labels=labels))
# df_final['OBS_AREA_TERCILE'] = pd.qcut(df_final['OBS_AREA'], n, labels=labels)

df_final_byfcstsize = df_final.groupby(['LEAD', 'INIT_MONTH', 'FCST_AREA_QUANTILE'])[vars+log_vars].mean().to_xarray().rename(
        {'LEAD': 'lead',
         'INIT_MONTH': 'month',
         'FCST_AREA_QUANTILE': 'category'}
    )

In [1]:
data = df_final_byfcstsize.mean('month')

# for i in range(n):
#     data['INTERSECTION_OVER_UNION'].isel(category=i).plot(c=colors[i], label=data['category'].values[i])
# # plt.legend()
# plt.show()
# for i in range(n):
#     data['CENTROID_DIST_KM'].isel(category=i).plot(c=colors[i], label=data['category'].values[i])
# # plt.legend()
# plt.show()
# for i in range(n):
#     np.exp(data['ABS_LOG_AREA_RATIO']).isel(category=i).plot(c=colors[i], label=data['category'].values[i])
# plt.yscale('log')
# # plt.legend()
# plt.show()
# for i in range(n):
#     np.exp(data['ABS_LOG_INTENSITY_50_RATIO']).isel(category=i).plot(c=colors[i], label=data['category'].values[i])
# plt.yscale('log')
# # plt.legend()
# plt.show()

In [60]:
data.to_netcdf('/glade/u/home/jtcohen/MODE/notebooks/GRL_code/final_data/attrs_by_area.nc')